# Module 9: Logging & Diagnostics

In this module, we will transition from using `print()` for debugging to using Python's professional `logging` library. 

### Goals for today:
1. Configure a global logger with specific levels.
2. Direct logs to a physical file on the hard drive.
3. Implement `RotatingFileHandler` to prevent disk overflow.

### 1. Basic Logging Configuration
Instead of printing everything, we define a "Threshold." If we set the threshold to **INFO**, all **DEBUG** messages will be ignored. This allows us to "turn down the noise" in production.

In [1]:
import logging

# Resetting any existing logging configuration (useful for re-running cells in a notebook)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# 1. Basic configuration: Set the level to DEBUG to see everything
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print("=== 1. Demonstrating Log Levels ===\n")

logging.debug("This is a DEBUG message (Hidden in production)")
logging.info("This is an INFO message (General updates)")
logging.warning("This is a WARNING (Potential problem detected)")
logging.error("This is an ERROR (A feature failed)")
logging.critical("This is a CRITICAL error (The system is crashing!)")

2026-04-22 14:19:46 - DEBUG - This is a DEBUG message (Hidden in production)
2026-04-22 14:19:46 - INFO - This is an INFO message (General updates)
2026-04-22 14:19:46 - WARNING - This is a WARNING (Potential problem detected)
2026-04-22 14:19:46 - ERROR - This is an ERROR (A feature failed)
2026-04-22 14:19:46 - CRITICAL - This is a CRITICAL error (The system is crashing!)


=== 1. Demonstrating Log Levels ===



### 2. Logging to a File
In a real-world system, logs need to be saved to a `.log` file so they can be reviewed later. We will now configure a logger that writes to both the console and a file.

In [2]:
# Create a dedicated logger for our application
logger = logging.getLogger("AppLogger")
logger.setLevel(logging.DEBUG)

# 1. Create a file handler to save logs to a file
file_handler = logging.FileHandler('application_debug.log')
file_handler.setLevel(logging.ERROR) # Only save Errors or higher to the file

# 2. Create a console handler to show updates in the terminal
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO) # Show Info and above in the terminal

# 3. Create a formatter and add it to the handlers
formatter = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

# 4. Add handlers to the logger
logger.addHandler(file_handler)
logger.addHandler(console_handler)

print("=== 2. Dual-Target Logging (Console vs File) ===\n")
logger.info("This will appear in the terminal ONLY.")
logger.error("This will appear in BOTH the terminal and the log file.")

# Verify the file was created and contains the error
with open('application_debug.log', 'r') as f:
    print(f"\nContents of 'application_debug.log':\n{f.read()}")

AppLogger - INFO - This will appear in the terminal ONLY.
2026-04-22 14:19:58 - INFO - This will appear in the terminal ONLY.
AppLogger - ERROR - This will appear in BOTH the terminal and the log file.
2026-04-22 14:19:58 - ERROR - This will appear in BOTH the terminal and the log file.


=== 2. Dual-Target Logging (Console vs File) ===


Contents of 'application_debug.log':
AppLogger - ERROR - This will appear in BOTH the terminal and the log file.



### 3. Implementing Rotating Logs
To prevent our log files from growing indefinitely, we use `RotatingFileHandler`. We will set a tiny limit (200 bytes) to force the logs to rotate quickly so you can see the effect.

In [3]:
from logging.handlers import RotatingFileHandler
import os

print("=== 3. Testing Log Rotation ===\n")

# Create a handler: Max 200 bytes, keep up to 3 old log files
rotator = RotatingFileHandler('rotating_system.log', maxBytes=200, backupCount=3)
logger.addHandler(rotator)

# Generate many log entries to fill up the 200-byte limit
for i in range(20):
    logger.info(f"Generated log entry number {i} to trigger rotation.")

# Check the directory to see the rotated files (.1, .2, etc.)
log_files = [f for f in os.listdir('.') if 'rotating_system.log' in f]
print(f"Log files found in directory: {log_files}")

AppLogger - INFO - Generated log entry number 0 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log entry number 0 to trigger rotation.
AppLogger - INFO - Generated log entry number 1 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log entry number 1 to trigger rotation.
AppLogger - INFO - Generated log entry number 2 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log entry number 2 to trigger rotation.
AppLogger - INFO - Generated log entry number 3 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log entry number 3 to trigger rotation.
AppLogger - INFO - Generated log entry number 4 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log entry number 4 to trigger rotation.
AppLogger - INFO - Generated log entry number 5 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log entry number 5 to trigger rotation.
AppLogger - INFO - Generated log entry number 6 to trigger rotation.
2026-04-22 14:20:24 - INFO - Generated log 

=== 3. Testing Log Rotation ===

Log files found in directory: ['rotating_system.log', 'rotating_system.log.1', 'rotating_system.log.2', 'rotating_system.log.3']


### Summary Checklist:
1. Replaced `print()` with `logging` for better diagnostics. (Yes)
2. Used different levels (DEBUG vs ERROR) to filter noise. (Yes)
3. Implemented rotation to protect the computer's hard drive. (Yes)